In [5]:
# Cell 1: Load combined normalized train/test from all nodes (A–H)
import pandas as pd
import os

nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
base_dir = "dataset/normalized"

train_files = [os.path.join(base_dir, f"Node_{node}_train_normalized.csv") for node in nodes]
test_files  = [os.path.join(base_dir, f"Node_{node}_test_normalized.csv")  for node in nodes]

print("Loading normalized train files:")
for p in train_files:
    print(" ", p)

print("\nLoading normalized test files:")
for p in test_files:
    print(" ", p)

df_train_list = [pd.read_csv(p) for p in train_files]
df_test_list  = [pd.read_csv(p) for p in test_files]

df_train = pd.concat(df_train_list, ignore_index=True)
df_test  = pd.concat(df_test_list,  ignore_index=True)

print("\n=== Normalized, combined datasets (Node A–H) ===")
print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)
print("\nColumns:", df_train.columns.tolist())
print("\nTarget value counts (train):")
print(df_train["Attack"].value_counts())
print("\nTarget value counts (test):")
print(df_test["Attack"].value_counts())

Loading normalized train files:
  dataset/normalized/Node_A_train_normalized.csv
  dataset/normalized/Node_B_train_normalized.csv
  dataset/normalized/Node_C_train_normalized.csv
  dataset/normalized/Node_D_train_normalized.csv
  dataset/normalized/Node_E_train_normalized.csv
  dataset/normalized/Node_F_train_normalized.csv
  dataset/normalized/Node_G_train_normalized.csv
  dataset/normalized/Node_H_train_normalized.csv

Loading normalized test files:
  dataset/normalized/Node_A_test_normalized.csv
  dataset/normalized/Node_B_test_normalized.csv
  dataset/normalized/Node_C_test_normalized.csv
  dataset/normalized/Node_D_test_normalized.csv
  dataset/normalized/Node_E_test_normalized.csv
  dataset/normalized/Node_F_test_normalized.csv
  dataset/normalized/Node_G_test_normalized.csv
  dataset/normalized/Node_H_test_normalized.csv

=== Normalized, combined datasets (Node A–H) ===
Train shape: (245856, 7)
Test shape:  (105456, 7)

Columns: ['shunt_voltage', 'bus_voltage_V', 'current_mA', '

In [6]:
# Cell 2: Train Random Forest on combined multi-node dataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score

# =============================================================================
# 1. Prepare features and target
# =============================================================================
target_col = "Attack"

# Define feature columns explicitly (adjust if you add/remove features)
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]

# Separate features and target
X_train_raw = df_train[feature_cols].copy()
y_train_raw = df_train[target_col].copy()

X_test_raw  = df_test[feature_cols].copy()
y_test_raw  = df_test[target_col].copy()

# One-hot encode 'State'
X_train = pd.get_dummies(X_train_raw, columns=["State"], drop_first=False)
X_test  = pd.get_dummies(X_test_raw,  columns=["State"], drop_first=False)

# Ensure same columns in train and test (in case some State value is missing in one split)
for col in ["State_idle", "State_charging"]:
    if col not in X_train.columns:
        X_train[col] = 0
    if col not in X_test.columns:
        X_test[col] = 0

# Enforce consistent column order
X_train = X_train[["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State_idle", "State_charging"]]
X_test  = X_test[["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State_idle", "State_charging"]]

# Encode target labels
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print("Encoded classes:", dict(zip(le.classes_, range(len(le.classes_)))))
print("Feature columns:", X_train.columns.tolist())
print("Training samples (all nodes):", X_train.shape[0])
print("Test samples (all nodes):", X_test.shape[0])

# =============================================================================
# 2. Scale numeric features (StandardScaler on numeric part only)
# =============================================================================
numeric_features = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features]  = scaler.transform(X_test[numeric_features])

# Convert to numpy for sklearn
X_train_np = X_train_scaled.to_numpy()
X_test_np  = X_test_scaled.to_numpy()

# =============================================================================
# 3. Train Random Forest with your chosen hyperparameters
# =============================================================================
rf_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)

rf_final.fit(X_train_np, y_train)

print("\nRandom Forest trained on combined multi-node dataset (Node A–H).")
print(f"Training samples: {X_train_np.shape[0]}, Features: {X_train_np.shape[1]}")
print(f"OOB score (train): {rf_final.oob_score_:.6f}")

Encoded classes: {'Backdoor': 0, 'none': 1, 'syn-flood': 2}
Feature columns: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_idle', 'State_charging']
Training samples (all nodes): 245856
Test samples (all nodes): 105456

Random Forest trained on combined multi-node dataset (Node A–H).
Training samples: 245856, Features: 6
OOB score (train): 0.920917


In [9]:
# Cell: Save trained RF model, LabelEncoder, and StandardScaler
import joblib
import os

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(rf_final, os.path.join(model_dir, "rf_final.joblib"))
joblib.dump(le,      os.path.join(model_dir, "label_encoder.joblib"))
joblib.dump(scaler,  os.path.join(model_dir, "scaler.joblib"))

print("Saved model and preprocessing objects to:", model_dir)
print("  - rf_final.joblib")
print("  - label_encoder.joblib")
print("  - scaler.joblib")

Saved model and preprocessing objects to: models
  - rf_final.joblib
  - label_encoder.joblib
  - scaler.joblib


In [7]:
# Cell 3: Per-node evaluation (Node A–H) using the model trained on all nodes
import os
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score, confusion_matrix

# Configuration
nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
base_dir = "dataset/normalized"
target_col = "Attack"
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]

# These must match what you used in Cell 2
expected_feature_order = [
    "shunt_voltage", "bus_voltage_V", "current_mA", "power_mW",
    "State_idle", "State_charging"
]
numeric_features = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

def prepare_node_df(df_raw, scaler, expected_cols):
    """
    Prepare a node dataframe for prediction:
      - select feature_cols
      - one-hot encode 'State'
      - ensure same columns as training
      - scale numeric features with the given scaler
      - return numpy array in expected column order
    """
    X_raw = df_raw[feature_cols].copy()
    
    # One-hot encode 'State'
    X = pd.get_dummies(X_raw, columns=["State"], drop_first=False)
    
    # Ensure both State columns exist
    for col in ["State_idle", "State_charging"]:
        if col not in X.columns:
            X[col] = 0
    
    # Enforce consistent column order
    X = X[expected_cols]
    
    # Scale numeric features
    X_scaled = X.copy()
    X_scaled[numeric_features] = scaler.transform(X[numeric_features])
    
    return X_scaled.to_numpy()

print("Per-node evaluation (model trained on all nodes A–H):\n")

for node in nodes:
    test_path = os.path.join(base_dir, f"Node_{node}_test_normalized.csv")
    print(f"\n=== Node {node} ===")
    print(f"Loading: {test_path}")
    
    df_test_node = pd.read_csv(test_path)
    
    # Features and target
    X_test_node_np = prepare_node_df(df_test_node, scaler, expected_feature_order)
    y_test_node_raw = df_test_node[target_col].copy()
    
    # Encode target using the same LabelEncoder as in training
    y_test_node = le.transform(y_test_node_raw)
    
    # Predictions
    y_test_node_pred = rf_final.predict(X_test_node_np)
    y_test_node_prob = rf_final.predict_proba(X_test_node_np)
    
    # Metrics
    macro_f1_node = f1_score(y_test_node, y_test_node_pred, average="macro")
    reconstruction_error_node = mean_squared_error(y_test_node, y_test_node_pred)
    test_loss_node = log_loss(y_test_node, y_test_node_prob)
    
    try:
        roc_auc_node = roc_auc_score(y_test_node, y_test_node_prob, multi_class="ovr", average="macro")
    except ValueError:
        # In case a node has only one class in test set
        roc_auc_node = np.nan
    
    print(f"Samples: {X_test_node_np.shape[0]}")
    print(f"  Macro_f1:             {macro_f1_node:.6f}")
    print(f"  Reconstruction_Error: {reconstruction_error_node:.6f}")
    print(f"  test_loss:            {test_loss_node:.6f}")
    print(f"  ROC_AUC:              {roc_auc_node:.6f}")
    
    # Confusion matrix
    cm_node = confusion_matrix(y_test_node, y_test_node_pred)
    print("Confusion Matrix (rows: true, cols: predicted):")
    print("Classes:", le.classes_)
    print(cm_node)

Per-node evaluation (model trained on all nodes A–H):


=== Node A ===
Loading: dataset/normalized/Node_A_test_normalized.csv
Samples: 13182
  Macro_f1:             0.842614
  Reconstruction_Error: 0.136246
  test_loss:            0.295517
  ROC_AUC:              0.957292
Confusion Matrix (rows: true, cols: predicted):
Classes: ['Backdoor' 'none' 'syn-flood']
[[1827 1211    4]
 [ 511 5066    0]
 [  10   18 4535]]

=== Node B ===
Loading: dataset/normalized/Node_B_test_normalized.csv
Samples: 13182
  Macro_f1:             0.842941
  Reconstruction_Error: 0.130784
  test_loss:            0.291932
  ROC_AUC:              0.954259
Confusion Matrix (rows: true, cols: predicted):
Classes: ['Backdoor' 'none' 'syn-flood']
[[1607  925    3]
 [ 745 5844    2]
 [   9    4 4043]]

=== Node C ===
Loading: dataset/normalized/Node_C_test_normalized.csv
Samples: 13182
  Macro_f1:             0.847072
  Reconstruction_Error: 0.121453
  test_loss:            0.290012
  ROC_AUC:              0.953050
Con

In [5]:
# perfect mixture between all 3 models and now the mixed is almost every time in the middle with the results